In [1]:
##################
### Keagan Moo ###
### 2024/06/04 ###
### PeakChipEn ###
##################

# This script loads in the bed files for age-associated and non age-associated peaks from a given cell type and runs chipenrich to test for enrichment of KEGG or GO gene sets in nearby genes to the
# age-associated peaks in comparison to the non age-associated ones. Also generates the heatmaps and barplots for these results.

suppressPackageStartupMessages({library(tidyverse)
library(glue)
library(data.table)
library(EnsDb.Hsapiens.v79)
library(chipenrich)
library("KEGGREST")
library("limma")
library("hash")
library("pheatmap")
library("ggpattern")
})


Warning message:
"package 'ggpattern' was built under R version 4.4.3"


In [2]:
citation("chipenrich")

To cite the R package 'chipenrich' in publications use:

  Welch RP, Lee C, Smith RA, Imbriano P, Scott LJ, Sartor MA.
  ChIP-Enrich: gene set enrichment testing for ChIP-seq data. Nucl.
  Acids Res. (2014) 42(13):e105. doi:10.1093/nar/gku463

A BibTeX entry for LaTeX users is

  @Article{,
    title = {ChIP-Enrich: Gene set enrichment testing for ChIP-Seq data},
    author = {Ryan P Welch and Chee Lee and Raymond G Cavalcante and Richard Alex Smith and Paul Imbriano and Laura J Scott and Maureen A Sartor},
    year = {2014},
    journal = {Nucleic Acids Research},
    note = {R package version 2.30.0},
  }

This free open-source software implements academic research by the
authors and co-workers. If you use it, please support the project by
citing the appropriate journal articles.

In [2]:
# Get backgrounds for your given nuclei type, this requires Male, female, and combined (?) backgrounds
cDex <- "Type_2a"
peakBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/{cDex}_bgo.bed"), sep = "\t")[-1,]
peakBedF <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/F_{cDex}_bgo.bed"), sep = "\t")[-1,]
peakBedM <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/M_{cDex}_bgo.bed"), sep = "\t")[-1,]
colnames(peakBed) <- c("chrom", "start", "end")
colnames(peakBedF) <- c("chrom", "start", "end")
colnames(peakBedM) <- c("chrom", "start", "end")
head(peakBedF)

,chrom,start,end
,<chr>,<chr>,<chr>
2,chr1,1000015,1000316
3,chr1,100037924,100038225
4,chr1,100132863,100133164
5,chr1,100178145,100178446
6,chr1,1001840,1002141
7,chr1,100187982,100188283


In [3]:
# Get selected peaks both combined and opening v closing to get a more comprehensive view. We have a lot fewer peaks with the new FDR < 0.1 method vs nominal p < 0.05
cBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/{cDex}_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(cBed) <- c("chrom", "start", "end")
cCloseBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/{cDex}_DOWN_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(cCloseBed) <- c("chrom", "start", "end")
cOpenBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/{cDex}_UP_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(cOpenBed) <- c("chrom", "start", "end")

fBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/F_{cDex}_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(fBed) <- c("chrom", "start", "end")
fCloseBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/F_{cDex}_DOWN_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(fCloseBed) <- c("chrom", "start", "end")
fOpenBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/F_{cDex}_UP_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(fOpenBed) <- c("chrom", "start", "end")

mBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/M_{cDex}_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(mBed) <- c("chrom", "start", "end")
mCloseBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/M_{cDex}_DOWN_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(mCloseBed) <- c("chrom", "start", "end")
mOpenBed <- read.table(file = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/M_{cDex}_UP_FDR05_bgo.bed"), sep = "\t")[-1,]
colnames(mOpenBed) <- c("chrom", "start", "end")

In [4]:
outPath <- "/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005/"
resPath <- "/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005/results/"

In [8]:
peakRes <- chipenrich(peakBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("{cDex}_GOMF_BG"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(peakRes$results, file = glue(resPath, "{cDex}_GOMF_BG_results.csv"))
peakResF <- chipenrich(peakBedF, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, #genesets = "/nfs/turbo/umms-scjp/kgmoo/data/GOMFLinks.tsv", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("F_{cDex}_GOMF_BG"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(peakResF$results, file = glue(resPath, "F_{cDex}_GOMF_BG_results.csv"))
peakResM <- chipenrich(peakBedM, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("M_{cDex}_GOMF_BG"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(peakResM$results, file = glue(resPath, "M_{cDex}_GOMF_BG_results.csv"))

Reading peaks from data.frame...

Warning message in postprocess_peak_grs(gr):
"Some input regions overlap. It is recommended that input regions be disjoint."
Assigning peaks to genes with assign_peaks(...) ..

Test: ChIP-Enrich

Genesets: Gene Ontology Molecular Function

Running tests..

Wrote results to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_GOMF_BG_results.tab

Wrote peak-to-gene assignments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_GOMF_BG_peaks.tab

Wrote run options/arguments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_GOMF_BG_opts.tab

Wrote count of peaks per gene to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_GOMF_BG_peaks-per-gene.tab

Wrote QC plots to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_GOMF_

In [5]:
cRes <- chipenrich(cBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("{cDex}_GOMF"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(cRes$results, file = glue(resPath, "{cDex}_GOMF_results005.csv"))
cCloseRes <- chipenrich(cCloseBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("{cDex}_GOMF_Close"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(cCloseRes$results, file = glue(resPath, "{cDex}_GOMF_Close_results005.csv"))
cOpenRes <- chipenrich(cOpenBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("{cDex}_GOMF_Open"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(cOpenRes$results, file = glue(resPath, "{cDex}_GOMF_Open_results005.csv"))

Reading peaks from data.frame...

Warning message in postprocess_peak_grs(gr):
"Some input regions overlap. It is recommended that input regions be disjoint."
Assigning peaks to genes with assign_peaks(...) ..

Test: ChIP-Enrich

Genesets: KEGG Pathways

Running tests..

Wrote results to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_KEGG_results.tab

Wrote peak-to-gene assignments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_KEGG_peaks.tab

Wrote run options/arguments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_KEGG_opts.tab

Wrote count of peaks per gene to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_KEGG_peaks-per-gene.tab

Wrote QC plots to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//Type_2a_KEGG_qcplots-1.png

Wrote QC plots t

In [7]:
fRes <- chipenrich(fBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("F_{cDex}_GOMF"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(fRes$results, file = glue(resPath, "F_{cDex}_GOMF_results005.csv"))
fCloseRes <- chipenrich(fCloseBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("F_{cDex}_GOMF_Close"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(fCloseRes$results, file = glue(resPath, "F_{cDex}_GOMF_Close_results005.csv"))
fOpenRes <- chipenrich(fOpenBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("F_{cDex}_GOMF_Open"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(fOpenRes$results, file = glue(resPath, "F_{cDex}_GOMF_Open_results005.csv"))


mRes <- chipenrich(mBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("M_{cDex}_GOMF"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(mRes$results, file = glue(resPath, "M_{cDex}_GOMF_results005.csv"))
mCloseRes <- chipenrich(mCloseBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("M_{cDex}_GOMF_Close"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(mCloseRes$results, file = glue(resPath, "M_{cDex}_GOMF_Close_results005.csv"))
mOpenRes <- chipenrich(mOpenBed, genome = "hg38", genesets = "GOMF", qc_plots = TRUE, 
                      out_path = outPath, out_name = glue("M_{cDex}_GOMF_Open"), min_geneset_size = 10, max_geneset_size = 1000)
write.csv(mOpenRes$results, file = glue(resPath, "M_{cDex}_GOMF_Open_results005.csv"))

Reading peaks from data.frame...

Assigning peaks to genes with assign_peaks(...) ..

Test: ChIP-Enrich

Genesets: KEGG Pathways

Running tests..

Wrote results to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_results.tab

Wrote peak-to-gene assignments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_peaks.tab

Wrote run options/arguments to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_opts.tab

Wrote count of peaks per gene to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_peaks-per-gene.tab

Wrote QC plots to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_qcplots-1.png

Wrote QC plots to: /nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/results/BedRes/CHIPEnrich005//F_Type_2x_KEGG_qcplots-2

In [59]:
temp <- fOpenRes$results
temp[temp$Geneset.ID == "hsa04150",]

,Geneset.Type,Geneset.ID,Description,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes
,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>
1,KEGG Pathways,hsa04150,mTOR signaling pathway,2.474543e-06,0.0005270776,2.304795,10.02212,enriched,52,5,139296.8,"5293, 5296, 54541, 57521, 64223"


In [17]:
cDex <- "Type_1"
#ResDF <- read.table(file = glue(resPath, "{cDex}_KEGG_Close_results.csv"), sep = ",", header = TRUE)
fResDF <- read.table(file = glue(resPath, "F_{cDex}_KEGG_Open_results.csv"), sep = ",", header = TRUE)
mResDF <- read.table(file = glue(resPath, "M_{cDex}_KEGG_Open_results.csv"), sep = ",", header = TRUE)
head(fResDF)

,X,Geneset.Type,Geneset.ID,Description,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes
,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>
1,1,KEGG Pathways,hsa04150,mTOR signaling pathway,2.474543e-06,0.0005270776,2.304795,10.022120,enriched,52,5,139296.8,"5293, 5296, 54541, 57521, 64223"
2,2,KEGG Pathways,hsa04960,Aldosterone-regulated sodium reabsorption,2.668541e-05,0.0028419960,2.278168,9.758789,enriched,42,4,138685.2,"3667, 4306, 5293, 5296"
3,3,KEGG Pathways,hsa03018,RNA degradation,4.661947e-05,0.0033099827,1.961327,7.108753,enriched,71,5,116372.6,"25804, 5073, 9125, 23404, 28960"
4,4,KEGG Pathways,hsa04910,Insulin signaling pathway,6.772588e-05,0.0033212321,1.633497,5.121752,enriched,138,7,125643.5,"31, 32, 572, 3667, 5293, 5296, 57521"
5,5,KEGG Pathways,hsa04930,Type II diabetes mellitus,7.796319e-05,0.0033212321,2.130995,8.423241,enriched,48,4,147797.0,"3667, 5293, 5296, 389692"
6,6,KEGG Pathways,hsa05213,Endometrial cancer,1.474900e-04,0.0052358934,2.046883,7.743723,enriched,52,4,200618.8,"572, 5293, 5296, 8313"


In [18]:
print("Male or Female")
CompDF <- merge(fResDF,mResDF, by = "Geneset.ID") %>% dplyr::select(Geneset.ID, Effect.x, Effect.y, P.value.x, P.value.y, FDR.x, FDR.y)
colnames(CompDF) <- c("Geneset.ID", "Effect.Female", "Effect.Male", "p.Female", "p.Male", "FDR.Female", "FDR.Male")
CompDF$modp.Female <- -log10(CompDF$p.Female)*sign(CompDF$Effect.Female)
CompDF$modp.Male <- -log10(CompDF$p.Male)*sign(CompDF$Effect.Male)

CompDF$ColorBlock <- rep("Niether", nrow(CompDF))
CompDF[CompDF$p.Female < 0.1,]$ColorBlock <- "Female"
CompDF[CompDF$p.Male < 0.1,]$ColorBlock <- "Male"
#CompDF[(CompDF$p.Female < 0.05) & (CompDF$p.Male < 0.05),]$ColorBlock <- "Both"
CompDF <- CompDF[CompDF$ColorBlock != "Niether",]
pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/enrich/CompareFemalevMaleSig_KEGGMMine_Open_Nom_{cDex}.pdf"), width=6, height=6)
    mygg <- ggplot(data=CompDF, aes(x=-log10(p.Female)*sign(Effect.Female), y=-log10(p.Male)*sign(Effect.Male))) +
        geom_point(aes(color = ColorBlock), alpha = 0.3)+#, position = position_jitter(width = 0.01)) +
        geom_abline(intercept = 0, slope = 1, alpha = 1, col = "black", linetype = "dashed") + geom_hline(yintercept = 0, color = "black") + geom_vline(xintercept = 0, color = "black") +
        theme_bw() + xlim(-11,11) + ylim(-11,11) + 
        xlab("KEGG Terms in Female Age Associated DARs -log10(Nominal P-value)*sign(coef)") + ylab("KEGG Terms in Male Age Associate DARs -log10(Nominal P-value)*sign(Coef)") + ggtitle(glue("Comparing Age Associated GO Terms Female v Male")) 
   print(mygg)
dev.off()

[1] "Male or Female"


pdf 
  2

In [14]:
nrow(fResDF[fResDF$P.value < 0.05,])
head(fResDF[fResDF$P.value < 0.05,])
nrow(mResDF[mResDF$P.value < 0.05,])
head(mResDF[mResDF$P.value < 0.05,])

[1] 12

,X,Geneset.Type,Geneset.ID,Description,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes
,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>
1,1,user-supplied,hsa04141,hsa04141,0.0003243334,0.1135167,1.2317261,3.427140,enriched,170,10,91271.13,"267, 1965, 3998, 5599, 6238, 10277, 64374, 80343, 91319, 201595"
2,2,user-supplied,hsa05216,hsa05216,0.0033240891,0.5817156,1.6122973,5.014317,enriched,37,4,129404.62,"595, 1647, 6258, 7849"
3,3,user-supplied,hsa00730,hsa00730,0.0164135208,0.9986290,1.9083327,6.741839,enriched,15,2,144335.20,"84284, 158067"
4,4,user-supplied,hsa04340,hsa04340,0.0173942438,0.9986290,1.1503935,3.159436,enriched,55,5,173250.00,"595, 2736, 55733, 57467, 57758"
5,5,user-supplied,hsa04330,hsa04330,0.0283320900,0.9986290,1.1780369,3.247992,enriched,61,4,124174.97,"7090, 6868, 7091, 23220"
6,6,user-supplied,hsa04666,hsa04666,0.0316206843,0.9986290,0.9360654,2.549929,enriched,95,6,140455.76,"1793, 3635, 8605, 10097, 50807, 57180"


[1] 13

,X,Geneset.Type,Geneset.ID,Description,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes
,<int>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>
1,1,user-supplied,hsa05322,hsa05322,3.805972e-05,0.01332090,2.656623,14.248094,enriched,126,3,44705.33,"8970, 8350, 8359"
2,2,user-supplied,hsa04979,hsa04979,1.261777e-04,0.02208110,2.918168,18.507344,enriched,51,2,85394.20,"1593, 114879"
3,3,user-supplied,hsa03320,hsa03320,6.403534e-04,0.07152693,2.589899,13.328421,enriched,74,2,77085.73,"1593, 2170"
4,4,user-supplied,hsa04613,hsa04613,8.174507e-04,0.07152693,2.110442,8.251889,enriched,180,3,76443.74,"8970, 8350, 8359"
5,5,user-supplied,hsa05034,hsa05034,1.177751e-03,0.08244259,2.036530,7.663970,enriched,179,3,87727.06,"8970, 8350, 8359"
6,6,user-supplied,hsa00120,hsa00120,1.848416e-03,0.10782429,3.305795,27.270217,enriched,17,1,88146.12,1593


In [15]:
nrow(fRes$results)
nrow(mRes$results)
nrow(mCloseRes$results)

[1] 350

[1] 350

[1] 350

In [5]:
# Aggregate and plot
T1ResComb <- read.table(file = glue(resPath, "Type_1_KEGG_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2aResComb <- read.table(file = glue(resPath, "Type_2a_KEGG_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2xResComb <- read.table(file = glue(resPath, "Type_2x_KEGG_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T1ResOpen <- read.table(file = glue(resPath, "Type_1_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2aResOpen <- read.table(file = glue(resPath, "Type_2a_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2xResOpen <- read.table(file = glue(resPath, "Type_2x_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T1ResClose <- read.table(file = glue(resPath, "Type_1_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2aResClose <- read.table(file = glue(resPath, "Type_2a_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))
T2xResClose <- read.table(file = glue(resPath, "Type_2x_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Type, Pattern))

T1ResFComb <- read.table(file = glue(resPath, "F_Type_1_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T1ResMComb <- read.table(file = glue(resPath, "M_Type_1_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResFComb <- read.table(file = glue(resPath, "F_Type_2a_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResMComb <- read.table(file = glue(resPath, "M_Type_2a_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResFComb <- read.table(file = glue(resPath, "F_Type_2x_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResMComb <- read.table(file = glue(resPath, "M_Type_2x_KEGG_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Combined") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))

T1ResFOpen <- read.table(file = glue(resPath, "F_Type_1_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T1ResMOpen <- read.table(file = glue(resPath, "M_Type_1_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResFOpen <- read.table(file = glue(resPath, "F_Type_2a_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResMOpen <- read.table(file = glue(resPath, "M_Type_2a_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResFOpen <- read.table(file = glue(resPath, "F_Type_2x_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResMOpen <- read.table(file = glue(resPath, "M_Type_2x_KEGG_Open_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Open") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))

T1ResFClose <- read.table(file = glue(resPath, "F_Type_1_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T1ResMClose <- read.table(file = glue(resPath, "M_Type_1_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_1") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResFClose <- read.table(file = glue(resPath, "F_Type_2a_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2aResMClose <- read.table(file = glue(resPath, "M_Type_2a_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2a") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResFClose <- read.table(file = glue(resPath, "F_Type_2x_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Female") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))
T2xResMClose <- read.table(file = glue(resPath, "M_Type_2x_KEGG_Close_results005.csv"), sep = ",", header = TRUE) %>%
            mutate(Sex = "Male") %>% mutate(Type = "Type_2x") %>% mutate(Pattern = "Close") %>% mutate(IDIG = paste(Geneset.ID, Sex, Type, Pattern))

LL <- list(T1ResFOpen, T1ResMOpen, T2aResFOpen, T2aResMOpen, T2xResFOpen, T2xResMOpen, T1ResFClose, T1ResMClose, T2aResFClose, T2aResMClose, T2xResFClose, T2xResMClose)
#LL <- list(T1ResFComb, T1ResMComb, T2aResFComb, T2aResMComb, T2xResFComb, T2xResMComb)
#LL <- list(T1ResOpen, T2aResOpen, T2xResOpen, T1ResClose, T2aResClose, T2xResClose)
TotalRes <- do.call("rbind", LL)
TotalRes$FDR_c <- p.adjust(TotalRes$P.value)
TotalRes$Description <- NULL

In [7]:
#T1ResFOpen[T1ResFOpen$Geneset.ID == "hsa04150",]
TotalRes[TotalRes$Geneset.ID == "hsa04150",]

,X,Geneset.Type,Geneset.ID,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes,Sex,Type,Pattern,IDIG,FDR_c
,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,1,KEGG Pathways,hsa04150,2.474543e-06,0.0005270776,2.3047946,1.002212e+01,enriched,52,5,139296.8,"5293, 5296, 54541, 57521, 64223",Female,Type_1,Open,hsa04150 Female Type_1 Open,0.006322457
333,120,KEGG Pathways,hsa04150,9.864318e-01,0.9905181069,-11.6886262,8.388690e-06,depleted,52,0,139296.8,,Male,Type_1,Open,hsa04150 Male Type_1 Open,1.000000000
474,48,KEGG Pathways,hsa04150,5.228658e-01,0.9913812253,0.4653011,1.592494e+00,enriched,52,2,139296.8,"6197, 54541",Female,Type_2a,Open,hsa04150 Female Type_2a Open,1.000000000
663,24,KEGG Pathways,hsa04150,1.518241e-01,0.9787845921,0.7567340,2.131304e+00,enriched,52,4,139296.8,"8408, 5296, 54541, 57521",Male,Type_2a,Open,hsa04150 Male Type_2a Open,1.000000000
871,19,KEGG Pathways,hsa04150,1.571255e-01,0.9882719636,1.4596897,4.304624e+00,enriched,52,1,139296.8,54541,Female,Type_2x,Open,hsa04150 Female Type_2x Open,1.000000000
1135,70,KEGG Pathways,hsa04150,7.132125e-01,0.9905620494,0.3745498,1.454337e+00,enriched,52,1,139296.8,6195,Male,Type_2x,Open,hsa04150 Male Type_2x Open,1.000000000
1386,108,KEGG Pathways,hsa04150,6.837751e-01,0.9998869208,-0.4149202,6.603930e-01,depleted,52,1,139296.8,253314,Female,Type_1,Close,hsa04150 Female Type_1 Close,1.000000000
1684,193,KEGG Pathways,hsa04150,9.843178e-01,0.9861946808,-12.0416748,5.893416e-06,depleted,52,0,139296.8,,Male,Type_1,Close,hsa04150 Male Type_1 Close,1.000000000
1783,79,KEGG Pathways,hsa04150,8.149235e-01,0.9866938294,0.1417013,1.152232e+00,enriched,52,3,139296.8,"3479, 7422, 253314",Female,Type_2a,Close,hsa04150 Female Type_2a Close,1.000000000


In [8]:
T1FBG <- read.table(file = glue(resPath, "F_Type_1_KEGG_BG_results.csv"), sep = ",", header = TRUE)
T1MBG <- read.table(file = glue(resPath, "M_Type_1_KEGG_BG_results.csv"), sep = ",", header = TRUE)
T2aFBG <- read.table(file = glue(resPath, "F_Type_2a_KEGG_BG_results.csv"), sep = ",", header = TRUE)
T2aMBG <- read.table(file = glue(resPath, "M_Type_2a_KEGG_BG_results.csv"), sep = ",", header = TRUE)
T2xFBG <- read.table(file = glue(resPath, "F_Type_2x_KEGG_BG_results.csv"), sep = ",", header = TRUE)
T2xMBG <- read.table(file = glue(resPath, "M_Type_2x_KEGG_BG_results.csv"), sep = ",", header = TRUE)

In [9]:
Thresh <- 0.05
overlistT1CombF <- intersect(T1FBG[T1FBG$FDR < Thresh,]$Geneset.ID, T1ResFComb[T1ResFComb$FDR < Thresh,]$Geneset.ID)
length(overlistT1CombF)
overlistT2aCombF <- intersect(T2aFBG[T2aFBG$FDR < Thresh,]$Geneset.ID, T2aResFComb[T2aResFComb$FDR < Thresh,]$Geneset.ID)
length(overlistT2aCombF)
overlistT2xCombF <- intersect(T2xFBG[T2xFBG$FDR < Thresh,]$Geneset.ID, T2xResFComb[T2xResFComb$FDR < Thresh,]$Geneset.ID)
length(overlistT2xCombF)

overlistT1CombM <- intersect(T1MBG[T1MBG$FDR < Thresh,]$Geneset.ID, T1ResMComb[T1ResMComb$FDR < Thresh,]$Geneset.ID)
length(overlistT1CombM)
overlistT2aCombM <- intersect(T2aMBG[T2aMBG$FDR < Thresh,]$Geneset.ID, T2aResMComb[T2aResMComb$FDR < Thresh,]$Geneset.ID)
length(overlistT2aCombM)
overlistT2xCombM <- intersect(T2xMBG[T2xMBG$FDR < Thresh,]$Geneset.ID, T2xResMComb[T2xResMComb$FDR < Thresh,]$Geneset.ID)
length(overlistT2xCombM)

overlist <- unique(c(overlistT1CombF, overlistT2aCombF, overlistT2xCombF,
                    overlistT1CombM, overlistT2aCombM, overlistT2xCombM))
length(overlist)
overlist

[1] 0

[1] 0

[1] 0

[1] 0

[1] 0

[1] 1

[1] 1

[1] "hsa04310"

In [10]:
Thresh <- 0.05
#head(T2aMBG[T2aMBG$FDR < 0.1,])
overlistT1OpenF <- intersect(T1FBG[T1FBG$FDR < Thresh,]$Geneset.ID, T1ResFOpen[T1ResFOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT1OpenF)
overlistT1CloseF <- intersect(T1FBG[T1FBG$FDR < Thresh,]$Geneset.ID, T1ResFClose[T1ResFClose$FDR < Thresh,]$Geneset.ID)
length(overlistT1CloseF)
overlistT2aOpenF <- intersect(T2aFBG[T2aFBG$FDR < Thresh,]$Geneset.ID, T2aResFOpen[T2aResFOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT2aOpenF)
overlistT2aCloseF <- intersect(T2aFBG[T2aFBG$FDR < Thresh,]$Geneset.ID, T2aResFClose[T2aResFClose$FDR < Thresh,]$Geneset.ID)
length(overlistT2aCloseF)
overlistT2xOpenF <- intersect(T2xFBG[T2xFBG$FDR < Thresh,]$Geneset.ID, T2xResFOpen[T2xResFOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT2xOpenF)
overlistT2xCloseF <- intersect(T2xFBG[T2xFBG$FDR < Thresh,]$Geneset.ID, T2xResFClose[T2xResFClose$FDR < Thresh,]$Geneset.ID)
length(overlistT2xCloseF)

overlistT1OpenM <- intersect(T1MBG[T1MBG$FDR < Thresh,]$Geneset.ID, T1ResMOpen[T1ResMOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT1OpenM)
overlistT1CloseM <- intersect(T1MBG[T1MBG$FDR < Thresh,]$Geneset.ID, T1ResMClose[T1ResMClose$FDR < Thresh,]$Geneset.ID)
length(overlistT1CloseM)
overlistT2aOpenM <- intersect(T2aMBG[T2aMBG$FDR < Thresh,]$Geneset.ID, T2aResMOpen[T2aResMOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT2aOpenM)
overlistT2aCloseM <- intersect(T2aMBG[T2aMBG$FDR < Thresh,]$Geneset.ID, T2aResMClose[T2aResMClose$FDR < Thresh,]$Geneset.ID)
length(overlistT2aCloseM)
overlistT2xOpenM <- intersect(T2xMBG[T2xMBG$FDR < Thresh,]$Geneset.ID, T2xResMOpen[T2xResMOpen$FDR < Thresh,]$Geneset.ID)
length(overlistT2xOpenF)
overlistT2xCloseM <- intersect(T2xMBG[T2xFBG$MDR < Thresh,]$Geneset.ID, T2xResMClose[T2xResMClose$FDR < Thresh,]$Geneset.ID)
length(overlistT2xCloseM)

overlist <- unique(c(overlistT1OpenF, overlistT1CloseF, overlistT2aOpenF, overlistT2aCloseF, overlistT2xOpenF, overlistT2xCloseF,
                    overlistT1OpenM, overlistT1CloseM, overlistT2aOpenM, overlistT2aCloseM, overlistT2xOpenM, overlistT2xCloseM))
length(overlist)
overlist

[1] 1

[1] 0

[1] 1

[1] 0

[1] 0

...

[1] 0

[1] 0

[1] 0

[1] 3

[1] "hsa04910" "hsa04010" "hsa05220"

In [11]:
KEGGDes <- data.frame(keggList("pathway", "hsa"))
colnames(KEGGDes)[1] <- "Description"
KEGGDes$Concept.Des <- str_split_fixed(KEGGDes$Description, " - ", 2)[,1]
KEGGDes$Geneset.ID <- rownames(KEGGDes)
KEGGDes$Description <- NULL
head(KEGGDes)
TotalRes <- merge(TotalRes, KEGGDes, by = "Geneset.ID")
nrow(TotalRes)
head(TotalRes)
min(TotalRes$FDR)

,Concept.Des,Geneset.ID
,<chr>,<chr>
hsa01100,Metabolic pathways,hsa01100
hsa01200,Carbon metabolism,hsa01200
hsa01210,2-Oxocarboxylic acid metabolism,hsa01210
hsa01212,Fatty acid metabolism,hsa01212
hsa01230,Biosynthesis of amino acids,hsa01230
hsa01232,Nucleotide metabolism,hsa01232


[1] 2544

,Geneset.ID,X,Geneset.Type,P.value,FDR,Effect,Odds.Ratio,Status,N.Geneset.Genes,N.Geneset.Peak.Genes,Geneset.Avg.Gene.Length,Geneset.Peak.Genes,Sex,Type,Pattern,IDIG,FDR_c,Concept.Des
,<chr>,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>
1,hsa00010,107,KEGG Pathways,0.9676577,0.9803342,-11.46833383,1.045601e-05,depleted,65,0,82434.29,,Female,Type_1,Close,hsa00010 Female Type_1 Close,1,Glycolysis / Gluconeogenesis
2,hsa00010,131,KEGG Pathways,0.9684183,0.9839744,-11.68257198,8.439631e-06,depleted,65,0,82434.29,,Male,Type_2x,Close,hsa00010 Male Type_2x Close,1,Glycolysis / Gluconeogenesis
3,hsa00010,23,KEGG Pathways,0.2367739,0.9912879,0.86375801,2.372058e+00,enriched,65,2,82434.29,"223, 83440",Male,Type_2a,Open,hsa00010 Male Type_2a Open,1,Glycolysis / Gluconeogenesis
4,hsa00010,84,KEGG Pathways,0.9582657,0.9897581,0.03808833,1.038823e+00,enriched,65,2,82434.29,"221, 5105",Male,Type_2a,Close,hsa00010 Male Type_2a Close,1,Glycolysis / Gluconeogenesis
5,hsa00010,49,KEGG Pathways,0.5999203,0.9763213,0.53436337,1.706362e+00,enriched,65,1,82434.29,226,Female,Type_2a,Open,hsa00010 Female Type_2a Open,1,Glycolysis / Gluconeogenesis
6,hsa00010,182,KEGG Pathways,0.9938594,0.9947530,-11.99683881,6.163666e-06,depleted,65,0,82434.29,NA,Female,Type_2x,Open,hsa00010 Female Type_2x Open,1,Glycolysis / Gluconeogenesis


[1] 0.003398377

In [12]:
SigList <- c("Temp")
for (tempFrame in LL) {
    if (!(nrow(tempFrame[tempFrame$FDR < Thresh,]) == 0)) {
        SigList <- c(SigList, tempFrame[tempFrame$FDR < Thresh,]$Geneset.ID) %>% unique()
    }
}
SigList <- SigList[-1]
SigList <- SigList[!(SigList %in% overlist)]
length(SigList)
TotalRes <- TotalRes[TotalRes$Geneset.ID %in% SigList,]
TotalRes$SetID <- paste0(TotalRes$Sex, "IXI", TotalRes$Type, "IXI", TotalRes$Pattern)
nrow(TotalRes)

[1] 4

[1] 48

In [13]:
stateList <- SigList
compList <- stateList[1]
setList <- unique(TotalRes$IDIG)
setOrder <- c("FemaleIXIType_1IXIOpen", "FemaleIXIType_1IXIClose", "FemaleIXIType_2aIXIOpen", "FemaleIXIType_2aIXIClose", "FemaleIXIType_2xIXIOpen", "FemaleIXIType_2xIXIClose",
             "MaleIXIType_1IXIOpen", "MaleIXIType_1IXIClose", "MaleIXIType_2aIXIOpen", "MaleIXIType_2aIXIClose", "MaleIXIType_2xIXIOpen", "MaleIXIType_2xIXIClose")
#setOrder <- c("FemaleIXIType_1IXICombined", "FemaleIXIType_2aIXICombined", "FemaleIXIType_2xIXICombined",
#             "MaleIXIType_1IXICombined", "MaleIXIType_2aIXICombined", "MaleIXIType_2xIXICombined")

#setOrder<- c("IXIType_1IXIOpen", "IXIType_1IXIOpen", "IXIType_1IXIOpen", 
#             "IXIType_1IXIOpen", "IXIType_1IXIOpen", "IXIType_1IXIOpen")

Thing <- TotalRes[TotalRes$Geneset.ID == stateList[1],] %>% arrange(match(SetID, setOrder))
stateHeatCo <- data.frame(Thing$Effect)
stateHeatPv <- data.frame(Thing$FDR)

sDex <- 0
for (state in stateList) {
    #print(state)
    tempThing <- TotalRes[TotalRes$Geneset.ID == state,] %>% arrange(match(SetID, setOrder))
    tempCo <- tempThing$Effect
    tempPv <- tempThing$FDR
    if((length(tempCo == 12))&(length(tempPv) == 12)) {
        stateHeatCo <- cbind(stateHeatCo, tempCo)
        stateHeatPv <- cbind(stateHeatPv, tempPv)
        compList <- c(compList, state)
    } else {
        sDex <- sDex + 1
        print(state)
        print(glue("Skipped total: {sDex}"))
    }
}

stateHeatCo <- stateHeatCo[,-1]
stateHeatPv <- stateHeatPv[,-1]
compList <- compList[-1]
TotalRes <- TotalRes[TotalRes$Geneset.ID %in% compList,]

colnames(stateHeatCo) <- compList
colnames(stateHeatPv) <- compList

refDF <- TotalRes %>% dplyr::select(c("Geneset.ID", "Concept.Des")) %>% distinct() %>% arrange(match(Geneset.ID, colnames(stateHeatCo)))
colnames(stateHeatCo) <- refDF$Concept.Des
colnames(stateHeatPv) <- refDF$Concept.Des

rownames(stateHeatCo) <- setOrder
rownames(stateHeatPv) <- setOrder

stateHeatCo <- stateHeatCo [ order(rownames(stateHeatCo)), ]
stateHeatPv <- stateHeatPv [ order(rownames(stateHeatPv)), ]

In [14]:
metaGlob <- rownames(stateHeatCo)
Translator <- hash()
Translator[["Female"]] <- "#FF0000"
Translator[["Male"]] <- "#0000FF"
Translator[["Type_1"]] <- "#e7a880"
Translator[["Type_2a"]] <- "#ce7331"
Translator[["Type_2x"]] <- "#b33c00"
Translator[["Closing"]] <- "#4AE4CB"
Translator[["Opening"]] <- "#CC6677"
metaList <- c()
for (metaBit in metaGlob) {
    metaRow <- strsplit(metaBit, "IXI")
    for (dDex in 1:length(metaRow[[1]])) {
      
      if (metaRow[[1]][dDex] %in% hash::keys(Translator)){
        
        trans <- metaRow[[1]][dDex]#Translator[[metaRow[[1]][dDex]]]
        metaRow[[1]][dDex] <- trans
        
      }
      
    }
    metaList <- c(metaList, metaRow)
    
}
metaFrame <- as.data.frame(t(as.data.frame(metaList)))
colnames(metaFrame) <- c("Sex","Cluster_Type", "Pattern")
rownames(metaFrame) <- metaGlob

metaFrame[metaFrame$Pattern == "Open",]$Pattern <- "Opening"
metaFrame[metaFrame$Pattern == "Close",]$Pattern <- "Closing"

metaFrame

,Sex,Cluster_Type,Pattern
,<chr>,<chr>,<chr>
FemaleIXIType_1IXIClose,Female,Type_1,Closing
FemaleIXIType_1IXIOpen,Female,Type_1,Opening
FemaleIXIType_2aIXIClose,Female,Type_2a,Closing
FemaleIXIType_2aIXIOpen,Female,Type_2a,Opening
FemaleIXIType_2xIXIClose,Female,Type_2x,Closing
FemaleIXIType_2xIXIOpen,Female,Type_2x,Opening
MaleIXIType_1IXIClose,Male,Type_1,Closing
MaleIXIType_1IXIOpen,Male,Type_1,Opening
MaleIXIType_2aIXIClose,Male,Type_2a,Closing


In [15]:
col_palH <- c("#fee100","#ffe43d","#ffe75c","#ffea75","#ffee8e","#fff1a5","#fff4bb","#fff8d2","#fffbe8","#ffffff",
              "#e8ecf6","#d2d9ed","#bbc6e4","#a4b3db","#8e9fd3","#778cca","#6079c1","#4a66b8","#3353af")
boundCol <- 3
brakers <- seq(-boundCol, boundCol, by = ((2*boundCol)/(length(col_palH))))
brakers
annoCol <- data.frame(PATTERN = metaFrame$Pattern, SEX = metaFrame$Sex, CELL_TYPE = metaFrame$Cluster_Type)
rownames(annoCol) <- rownames(metaFrame)
annoColor <- list(PATTERN = c(Closing = "#4AE4CB", Opening = "#CC6677"),
                  SEX = c(Male = "#0000FF", Female = "#FF0000"), 
    CELL_TYPE = c(Type_1 = "#e7a880", Type_2a = "#ce7331", Type_2x = "#b33c00")
                 )

[1] -3.0000000 -2.6842105 -2.3684211 -2.0526316 -1.7368421 -1.4210526
 [7] -1.1052632 -0.7894737 -0.4736842 -0.1578947  0.1578947  0.4736842
[13]  0.7894737  1.1052632  1.4210526  1.7368421  2.0526316  2.3684211
[19]  2.6842105  3.0000000

In [18]:
stateHeatUnt <- as.matrix(t(stateHeatCo))
stateHeatLab <- as.matrix(t(stateHeatPv))
stateHeatLab[stateHeatLab < 0.05] <- "*"
stateHeatLab[stateHeatLab >= 0.05] <- ""
pheatmap(stateHeatUnt, 
                    col = rev(col_palH),
                    #cluster_rows = FALSE,
                    cluster_cols = FALSE,
                    breaks = brakers,
                    #cexCol = 1, 
                    annotation_col = annoCol,
                    annotation_colors = annoColor,
                    #cutree_rows = 2,
                    treeheight_row = 0,
                    #cutree_cols = 2,
                    #treeheight_col = 16,
                    display_numbers = stateHeatLab,
                    fontsize_number=16,
                    number_color = "black",
                    filename = glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/enrich/Fiber_chipenrich_OC_KEGG_FDR005_verbose_Test.pdf"),
                    #border_color=NA,
                    show_colnames = F,
                    #show_rownames = F,
                    main=glue("DAR Set by KEGG terms"),
                    width = 20, height = 5.5, fontsize = 20,
                    )

In [54]:
selectConcept <- c("Type II diabetes mellitus")#"Adipocytokine pathways", "Apoptosis", "Regulation of actin cytoskeleton",
                                                      #"Arrythmogenic right ventricular cardiomyopathy", "Type II diabetes mellitus"
selectFrame <- TotalRes[TotalRes$Concept.Des %in% selectConcept,]
#head(selectFrame)
labelFrame <- selectFrame[selectFrame$FDR < 0.1, ] %>% dplyr::select(SetID, Effect, Sex, Pattern, Type)
labelFrame[labelFrame$Effect < 0,]$Effect <- labelFrame[labelFrame$Effect < 0,]$Effect - 0.04
labelFrame[labelFrame$Effect > 0,]$Effect <- labelFrame[labelFrame$Effect > 0,]$Effect + 0.02

ColorCode_Female <- "#FF2222"
ColorCode_Male <- "#2222FF"

#selectFrame <- selectFrame %>% dplyr::filter(Type == "Type_1")

pdf(glue("/nfs/turbo/umms-scjp/kgmoo/analysis/20250428_kgmoo_ATACge/KMScripts/Plots/enrich/ATACEnrich_{selectConcept}_Bar.pdf"), width=6, height=6)
mygg <- ggplot(selectFrame, aes(x = SetID, y = Effect, fill = Sex, pattern = Pattern)) +
  geom_bar_pattern(position = 'dodge', stat = "identity", pattern_fill = "black", pattern_color = "black", color = "black") +
  xlab("Fiber type by sex") + ylab("Enrichment of Term") + ggtitle(glue("{selectConcept}")) + 
  scale_fill_manual(values = c(ColorCode_Female,ColorCode_Male)) +
  scale_pattern_manual(values = c(Open = "none", Close = "stripe")) +
  scale_y_continuous(breaks=seq(-3.5,3.5,0.5), limits = c(-3.5,3.5)) + theme_bw() +
  theme(axis.text.x = element_text(angle = 45, vjust = 1, hjust=1)) +
  guides(pattern = guide_legend(override.aes = list(fill = "white")),
         fill = guide_legend(override.aes = list(pattern = "none")))+ 
  geom_text(data = labelFrame, label = "*", size=8) 
mygg
dev.off()

Warning message:
"Removed 5 rows containing missing values or values outside the scale range
(`geom_bar_pattern()`)."


pdf 
  2